# Chapter 3 – Collaborative Filtering with ALS

This notebook implements Sections 3.3.3 through 3.3.9:
- Converting explicit ratings to implicit feedback
- Building a sparse user-item matrix
- Training an ALS model with the `implicit` library
- Generating user-based and item-based recommendations
- Using the four-stage framework

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from recsys.data.loaders import load_movielens
from recsys.utils.colab import get_data_path

In [ ]:
%pip install implicit

## 1. Load data and convert to implicit feedback (Listing 3.2)

In [ ]:
DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)

print(f"Loaded {len(ratings):,} ratings")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['movieId'].nunique():,}")

In [ ]:
# Listing 3.2: Converting explicit ratings to implicit feedback
implicit_ratings = ratings[ratings['rating'] >= 4].copy()  #A
implicit_ratings['confidence'] = implicit_ratings['rating']  #B
implicit_ratings['rating'] = 1  #C

#A Keep only ratings 4 and 5 (positive feedback)
#B Use original rating as confidence weight (5-star > 4-star)
#C Binary indicator: user liked this item

print(f"Original ratings: {len(ratings):,}")
print(f"High ratings (4-5 stars): {len(implicit_ratings):,}")
print(f"Conversion rate: {len(implicit_ratings)/len(ratings)*100:.1f}%")

## 2. Build sparse user-item matrix

The `create_user_item_matrix` helper lives in `recsys.data.sparse`.
It returns the CSR matrix plus mapping dictionaries.

In [ ]:
from recsys.data.sparse import create_user_item_matrix

user_item_matrix, user_map, item_map = create_user_item_matrix(
    implicit_ratings,
    user_col='userId',
    item_col='movieId',
    value_col='confidence'
)

print(f"Matrix shape: {user_item_matrix.shape}")
print(f"Non-zero entries: {user_item_matrix.nnz:,}")
print(f"Sparsity: {100 * (1 - user_item_matrix.nnz / np.prod(user_item_matrix.shape)):.2f}%")
print(f"Memory (sparse): {user_item_matrix.data.nbytes / (1024**2):.1f} MB")

## 3. Train the ALS model (Listing 3.4)

In [ ]:
# Listing 3.4: Training an ALS model
from implicit.als import AlternatingLeastSquares
import time

model = AlternatingLeastSquares(
    factors=50,           #A
    iterations=20,        #B
    regularization=0.01,  #C
    random_state=42       #D
)

#A Dimensionality of latent factor vectors
#B Number of alternating optimization rounds
#C Regularization to prevent overfitting
#D Reproducible results

start_time = time.time()
model.fit(user_item_matrix)
elapsed = time.time() - start_time

print(f"Training completed in {elapsed:.1f} seconds")
print(f"User factors: {model.user_factors.shape}")
print(f"Item factors: {model.item_factors.shape}")

## 4. Generate recommendations using the framework (Listings 3.5–3.9)

The `ALSRetrieval` class wraps the implicit model in the four-stage
framework introduced in Chapter 2. It handles both user-based and
seed-based retrieval.

In [ ]:
from recsys.fourstage_recsys.retrieval.als_retrieval import ALSRetrieval
from recsys.fourstage_recsys.recsys_context import RecommendationContext

recommender = ALSRetrieval(
    model=model,
    user_item_matrix=user_item_matrix,
    user_map=user_map,
    item_map=item_map,
    k=100
)

### User-based recommendations (Listing 3.8)

In [ ]:
def print_recommendations(recommendations, movies_df, top_n=10):
    """Display recommendations with titles and scores."""
    for i, rec in enumerate(recommendations[:top_n], 1):
        movie = movies_df[movies_df['movieId'] == rec['item_id']]
        if not movie.empty:
            title = movie.iloc[0]['title']
            score = rec['als_cf']
            print(f"{i}. {title} (score: {score:.3f})")


def print_user_history(user_id, ratings_df, movies_df, min_rating=4):
    """Show what a user has rated highly."""
    user_ratings = ratings_df[
        (ratings_df['userId'] == user_id) &
        (ratings_df['rating'] >= min_rating)
    ].merge(movies_df[['movieId', 'title']], on='movieId')
    user_ratings = user_ratings.sort_values('rating', ascending=False)
    print(f"User {user_id} highly-rated movies:")
    for _, row in user_ratings.head(10).iterrows():
        print(f"  {row['title']} (rating: {row['rating']})")

In [ ]:
# Listing 3.8: Generating recommendations for a user
user_id = "54"
context = RecommendationContext(user_id=user_id)
recommendations = recommender.retrieve(context)

print(f"Recommendations for User {user_id}:\n")
print_recommendations(recommendations, movies)

In [ ]:
# What did User 54 actually watch?
print_user_history(int(user_id), ratings, movies)

### Item-based (seed) recommendations (Listing 3.9)

In [ ]:
# Listing 3.9: Finding similar items
context = RecommendationContext(seed_items=["1"])  # Toy Story
recommendations = recommender.retrieve(context)

print("Movies similar to Toy Story (1995):\n")
print_recommendations(recommendations, movies)

### Multiple seed items (Section 3.3.6)

The `ALSRetrieval.with_seeds` method uses the multi-query union pattern:
retrieve neighbors for each seed separately, then merge by keeping
the max similarity score.

In [ ]:
# Seeds from different genres
context = RecommendationContext(seed_items=["1", "593"])  # Toy Story + Silence of the Lambs
recommendations = recommender.retrieve(context)

print("Recommendations from mixed seeds (Toy Story + Silence of the Lambs):\n")
print_recommendations(recommendations, movies)

## 5. Hyperparameter exploration (Section 3.3.8)

How does the number of iterations affect recommendations?

In [ ]:
from implicit.als import AlternatingLeastSquares

model_iter = AlternatingLeastSquares(
    factors=50,
    iterations=1,
    regularization=0.01,
    random_state=42
)

user_54_idx = user_map["54"]

for iteration in range(1, 41):
    model_iter.fit(user_item_matrix, show_progress=False)

    if iteration % 5 == 0:
        ids, scores = model_iter.recommend(
            user_54_idx,
            user_item_matrix[user_54_idx],
            N=5,
            filter_already_liked_items=True
        )
        reverse_item_map = {v: k for k, v in item_map.items()}
        top_id = reverse_item_map[ids[0]]
        top_title = movies[movies['movieId'] == top_id]['title'].values[0]
        print(f"Iteration {iteration:2d}: top score = {scores[0]:.3f}, "
              f"top movie = {top_title}")